# Figure 3: Mutation-prediction scores relate to mutation consequence

Reads the TCGA run outputs from `Runs/TCGA/output/cpm/` and the per-cohort MC3 event files from `Runs/TCGA/data/mutation_events/`, and writes panels to `Figures/Figure3/output/`.

### Main panels
- **a**: prediction scores by mutation-consequence class (Kruskal-Wallis)
- **b**: gene-cancer paired high-impact vs missense
- **c**: within-sample paired silent vs WT
- **d**: within-sample permutation test for the silent-WT comparison

### Supplemental
- gene-cancer paired silent vs WT
- sample-level high-impact vs missense
- PolyPhen and SIFT splits within missense variants

All panels are restricted to genes with normalized AUPRC > 0.1.

In [ ]:
import sys
from pathlib import Path

# ---------- path setup ----------
NOTEBOOK_DIR = Path.cwd().resolve()
FIGURES_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = FIGURES_DIR.parent
for path in (NOTEBOOK_DIR, FIGURES_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# ---------- shared style ----------
from constants import ORANGE, TEAL, PURPLE, ANNOTATION_FS

# ---------- Figure 3 helpers ----------
from figure3_helpers import (
    MUT_COLOR_WT, MUT_COLOR_SILENT, MUT_COLOR_MISSENSE, MUT_COLOR_HIGH_IMPACT,
    load_legacy_mc3_event_maf,
    load_predictions,
    load_auprc_metrics,
    build_pred_long,
    assign_mutation_effects,
    build_silent_only_df,
    build_missense_only_df,
    parse_polyphen,
    parse_sift,
    plot_pred_by_mut_effect,
    plot_paired_gc_panel,
    plot_paired_sample_panel,
    plot_score_split_panel,
    run_gc_permutations,
    plot_gc_permutation_histogram,
)

# ---------- data / output paths ----------
TCGA_RUN             = PROJECT_ROOT / "Runs" / "TCGA"
MUTATION_RESULTS_DIR = TCGA_RUN / "output" / "cpm"
MC3_EVENT_DIR        = TCGA_RUN / "data" / "mutation_events"
FIGURE_ROOT          = NOTEBOOK_DIR / "output"
SUPPLEMENT_ROOT      = FIGURE_ROOT / "Supplemental"

NORMALIZED_AUPRC_THRESHOLD = 0.1

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENT_ROOT.mkdir(parents=True, exist_ok=True)

## Load predictions and AUPRC metrics

In [ ]:
predictions = load_predictions(MUTATION_RESULTS_DIR)
auprc_df, prevalence_df, normalized_auprc_df = load_auprc_metrics(MUTATION_RESULTS_DIR)
print(f"{len(predictions)} cancer types loaded")
print(f"AUPRC table: {auprc_df.shape}")

## Long-form predictions (well-predicted genes only)

In [ ]:
pred_long = build_pred_long(
    predictions, normalized_auprc_df,
    normalized_auprc_threshold=NORMALIZED_AUPRC_THRESHOLD,
)
print(f"pred_long shape: {pred_long.shape}")
pred_long.head()

## Per-cohort MC3 event files and mutation-consequence classes

In [ ]:
maf = load_legacy_mc3_event_maf(
    MC3_EVENT_DIR,
    sorted(predictions),
    pass_only=True,
)
print(f"Mutation rows from per-cohort MC3 event files: {len(maf):,}")
print(f"Event files loaded: {maf['source_file'].nunique():,}")
print(f"Samples: {maf['sample4'].nunique():,}")
print(f"Genes: {maf['Hugo_Symbol'].nunique():,}")
print(maf['mutation_effect_class'].value_counts(dropna=False).to_string())
maf.head()

In [ ]:
mutation_effects_df = assign_mutation_effects(
    pred_long, maf,
    keep_other_as_separate=False,
    drop_other_mutations=True,
)
mutation_effects_df.head()

## Panel a: prediction scores by mutation-consequence class

In [ ]:
fig_a, ax_a = plot_pred_by_mut_effect(
    mutation_effects_df,
    save_path=FIGURE_ROOT / "figure3a_violin_mut_effect.pdf",
)

## Silent-only and missense-only datasets

In [ ]:
silent_df = build_silent_only_df(maf, pred_long)
silent_df.head()

In [ ]:
missense_df = build_missense_only_df(maf, pred_long)
missense_df.head()

## Panel b: gene-cancer paired high-impact vs missense (+ supplemental sample-level)

In [ ]:
panel_missense_hi = mutation_effects_df.loc[
    mutation_effects_df["mut_effect"].isin(["missense", "high_impact"])
].copy()

missense_hi_gc_result = plot_paired_gc_panel(
    panel_missense_hi,
    group_col="mut_effect",
    group_a="missense",
    group_b="high_impact",
    label_a="Missense",
    label_b="High impact",
    color_a=MUT_COLOR_MISSENSE,
    color_b=MUT_COLOR_HIGH_IMPACT,
    delta_color=PURPLE,
    save_path=FIGURE_ROOT / "figure3b_gc_highimpact_vs_missense.pdf",
)

missense_hi_sample_result = plot_paired_sample_panel(
    panel_missense_hi,
    group_a="missense",
    group_b="high_impact",
    label_a="Missense",
    label_b="High impact",
    color_a=MUT_COLOR_MISSENSE,
    color_b=MUT_COLOR_HIGH_IMPACT,
    delta_color=PURPLE,
    save_path=SUPPLEMENT_ROOT / "Supp_sample_highimpact_vs_missense.pdf",
)

## Panel c: within-sample paired silent vs WT (+ supplemental gene-cancer level)

In [ ]:
panel_b_wt_silent = mutation_effects_df.loc[
    mutation_effects_df["mut_effect"].isin(["no_mutation", "silent"])
].copy()

gc_result = plot_paired_gc_panel(
    panel_b_wt_silent,
    group_col="mut_effect",
    group_a="no_mutation",
    group_b="silent",
    label_a="WT",
    label_b="Silent",
    color_a=MUT_COLOR_WT,
    color_b=MUT_COLOR_SILENT,
    delta_color=PURPLE,
    save_path=SUPPLEMENT_ROOT / "Supp_gc_silent_vs_wt.pdf",
)

sample_result = plot_paired_sample_panel(
    panel_b_wt_silent,
    group_a="no_mutation",
    group_b="silent",
    label_a="WT",
    label_b="Silent",
    color_a=MUT_COLOR_WT,
    color_b=MUT_COLOR_SILENT,
    delta_color=PURPLE,
    save_path=FIGURE_ROOT / "figure3c_sample_silent_vs_wt.pdf",
)

## Panel d: within-sample permutation test (WT vs silent)

Shuffles WT/silent labels within each sample 1,000 times and recomputes the gene-cancer mean delta.

In [ ]:
perm_test = run_gc_permutations(
    panel_b_wt_silent,
    n_perm=1000,
    seed=1,
    n_jobs=-1,   # use all cores; set n_jobs=1 if joblib freezes in this environment
)
perm_test["perm_summary"].describe()

In [ ]:
fig_perm, ax_perm, perm_p = plot_gc_permutation_histogram(
    perm_test,
    stat_col="mean_delta",
    alternative="greater",
    save_path=FIGURE_ROOT / "figure3d_permutation_wt_silent.pdf",
)
print(f"Permutation p-value: {perm_p:.2e}")

## Supplemental: PolyPhen score (missense deleteriousness)

Higher PolyPhen score → more likely damaging. Expect genes with high PolyPhen to have higher
prediction scores.

In [ ]:
missense_df["polyphen_score"] = missense_df["PolyPhen"].apply(parse_polyphen)
print(f"PolyPhen coverage: {missense_df['polyphen_score'].notna().mean():.1%}")
missense_df["polyphen_score"].describe()

In [ ]:
missense_df['Cancer'] = missense_df['Cancer_x']

In [ ]:
polyphen_result = plot_score_split_panel(
    missense_df,
    score_col="polyphen_score",
    score_name="PolyPhen",
    direction="high_minus_low",
    color_low=MUT_COLOR_SILENT,
    color_high=MUT_COLOR_HIGH_IMPACT,
    delta_color=PURPLE,
    save_path=SUPPLEMENT_ROOT / "Supp_polyphen_split.pdf",
)

## Supplemental: SIFT score (missense deleteriousness)

Lower SIFT score → more deleterious. Expect genes with low SIFT to have higher prediction scores.

In [ ]:
missense_df["sift_score"] = missense_df["SIFT"].apply(parse_sift)
print(f"SIFT coverage: {missense_df['sift_score'].notna().mean():.1%}")
missense_df["sift_score"].describe()

In [ ]:
sift_result = plot_score_split_panel(
    missense_df,
    score_col="sift_score",
    score_name="SIFT",
    direction="low_minus_high",
    color_low=MUT_COLOR_HIGH_IMPACT,
    color_high=MUT_COLOR_SILENT,
    delta_color=PURPLE,
    save_path=SUPPLEMENT_ROOT / "Supp_sift_split.pdf",
)

## Summary

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    # Main panels
    {"panel": "c — sample-level silent vs WT",          "n": sample_result["pooled_n"],            "p": sample_result["pooled_p"]},
    {"panel": "d — permutation (within-sample null)",   "n": len(perm_test["perm_summary"]),        "p": perm_p},
    {"panel": "b — gc high-impact vs missense",          "n": missense_hi_gc_result["pooled_n"],    "p": missense_hi_gc_result["pooled_p"]},
    # Supplemental
    {"panel": "Supp — gc silent vs WT",                  "n": gc_result["pooled_n"],                "p": gc_result["pooled_p"]},
    {"panel": "Supp — sample high-impact vs missense",   "n": missense_hi_sample_result["pooled_n"],"p": missense_hi_sample_result["pooled_p"]},
    {"panel": "Supp — PolyPhen split",                   "n": polyphen_result["pooled_n"],          "p": polyphen_result["pooled_p"]},
    {"panel": "Supp — SIFT split",                       "n": sift_result["pooled_n"],              "p": sift_result["pooled_p"]},
])
print(summary.to_string(index=False))